# Compliance Streaming — Evidence Notebook

## Purpose (Read-only Evidence)
This notebook exists **only to prove claims** about the Compliance Streaming system using **already-generated artifacts**.

It does **not**:
- run pipelines or watchers
- regenerate events
- recompute metrics beyond simple read-only counts
- modify any files

Each evidence “exhibit” is always written as **3 consecutive cells**:

1) **Explanation (Markdown)**  
- What claim is being proven  
- Which artifact(s) will be inspected  
- What would invalidate the claim

2) **Execution (Read-only Code)**  
- Only `print()`, file reads, JSON parsing, line counts, previews  
- No mutation, no re-runs, no “fixing”, no parameter changes

3) **Evidence (Markdown)**  
- What the displayed output proves (pass/fail)  
- No narrative beyond what is shown

---

## Declared Guarantees Under Review
This notebook evaluates evidence for:

- Persisted events in the append-only stream are **not skipped**
- Contract violations **fail-fast** with **no coercion**
- Rule evaluation is **deterministic** and window-based on `event_time`
- Restarts resume from the **last checkpointed byte offset**

---

## Explicit Non-Support (Boundaries)
The following are explicitly not supported and are not evaluated:

- Events not yet written to disk (pre-persistence loss boundary)
- Cross-process or cross-run deduplication (dedupe is process-scoped only)
- Exactly-once beyond a single watcher process boundary


## Section 1 — Run Log

### Explanation

This section inspects the **run log** to establish the **execution context** of the watcher.

The claim being proven:

- A watcher run occurred and produced an authoritative run record
- The run log captures immutable execution metadata (run identifier, timing, configuration context)

Artifact under inspection:

- `logs/run_<run_id>.json`

What this section **proves**:
- That this evidence notebook refers to a real watcher execution
- The temporal boundaries of that execution

What this section **does not prove**:
- Restart correctness
- Event completeness
- Rule correctness or outcomes

Those are proven in later sections using other artifacts.


In [10]:
# Section 1 — Run Log

import json
import pandas as pd

with open("logs/run_ddff9dc1c1b4446bbfbf93aa76a9aceb.json") as f:
    run_log = json.load(f)

pd.DataFrame.from_dict(run_log, orient="index", columns=["value"])


,value
run_id,ddff9dc1c1b4446bbfbf93aa76a9aceb
started_at,2026-01-31T20:37:05.255603+00:00
ended_at,2026-01-31T20:37:08.075610+00:00
stream_path,data/stream.log
flags_path,outputs/flags/flags.log
final_offset,8299535
contract,"{'name': 'money_flow_event_contract', 'version..."
counts,"{'events_read': 788, 'events_valid': 788, 'fla..."
status,success


### Evidence

The run log confirms that a **single watcher execution** occurred and completed successfully.

**Proves**
- A watcher run existed (`run_id` present)
- The run has clear temporal boundaries (`started_at`, `ended_at`)
- The stream and output artifact paths used in this run
- The contract name and version applied
- Final processing status (`success`)
- Aggregate counts reported by the watcher at shutdown

**Does NOT prove**
- Restart correctness
- Offset persistence across restarts
- Event completeness or ordering
- Rule correctness or flag validity

Those claims are evaluated in subsequent sections using checkpoints, flags, and validation artifacts.


## Section 2 — Stream Log (Persisted Input)

### Explanation

This section inspects the **persisted event stream** to establish the **input boundary** for the watcher.

The claim being proven:

- Events written to the append-only stream file form the authoritative input set
- The watcher processes only events that exist in this persisted file

Artifact under inspection:

- `data/stream.log`

What this section **proves**:
- That a persisted input stream exists
- The physical size and structure of the input boundary

What this section **does not prove**:
- That all events were processed
- Ordering or duplication guarantees
- Rule evaluation correctness

Those are proven using checkpoints and validation artifacts.


In [12]:
# Section 2 — Stream Log

import pandas as pd

stream_path = "data/stream.log"

# Read a small, fixed preview for inspection only
df = pd.read_json(stream_path, lines=True, nrows=5)

df


,event_id,user_id,event_time,ingest_time,event_type,amount,currency,channel
0,c2837b9b-d20d-4cdc-a5e9-5cbf46833d56,user_00433,2026-01-31 20:32:00.889613+00:00,2026-01-31 20:32:00.889897+00:00,withdrawal_requested,21.84,GBP,api
1,6fa1b78b-b3d2-4762-917b-c99fff517084,user_00008,2026-01-31 20:32:00.900172+00:00,2026-01-31 20:32:00.900175+00:00,deposit_completed,249.91,GBP,mobile
2,010adf68-8029-406b-a5ce-682dfa6e158b,user_09637,2026-01-31 20:32:00.910359+00:00,2026-01-31 20:32:00.910365+00:00,deposit_completed,488.34,GBP,api
3,8ac4b448-9a41-49bb-b22f-664fe81838fd,user_07172,2026-01-31 20:32:00.920557+00:00,2026-01-31 20:32:00.920563+00:00,deposit_completed,88.20,GBP,api
4,1d8b0df5-2ec0-4bd6-bd0a-2f62067b2652,user_06916,2026-01-31 20:32:00.930749+00:00,2026-01-31 20:32:00.930756+00:00,deposit_completed,173.13,GBP,api


### Evidence

The displayed rows confirm that a **persisted append-only stream file** exists and contains structured JSON event records.

**Proves**
- `data/stream.log` exists and is readable
- Events are stored as line-delimited JSON records
- The watcher’s input is based on persisted data, not in-memory events

**Does NOT prove**
- That all persisted events were processed
- Event ordering or duplicate handling
- Rule evaluation or flag generation
- Restart correctness

Those claims are evaluated using checkpoint and validation artifacts.


## Section 3 — Checkpoint

### Explanation

This section inspects the **checkpoint artifact** to prove that the watcher
persisted its **last processed byte offset** to disk.

The claim being proven:

- A checkpoint file exists
- The checkpoint records a concrete byte `offset`
- The offset represents the persisted restart position at shutdown

Artifact under inspection:

- `outputs/checkpoints/state.json`

What this section **proves**:
- Restart position is explicitly persisted
- Restart safety exists within the persisted-stream boundary

What this section **does not prove**:
- Cross-run or cross-process exactly-once guarantees
- That no events were lost before persistence
- Rule correctness or validation outcomes


In [14]:
# Section 3 — Checkpoint

import json
import pandas as pd

with open("outputs/checkpoints/state.json") as f:
    checkpoint = json.load(f)

pd.DataFrame.from_dict(checkpoint, orient="index", columns=["value"])


,value
offset,8299535
saved_at,2026-01-31T20:37:08.075267+00:00


### Evidence

The checkpoint artifact confirms that the watcher persisted its
**restart position** at shutdown.

**Proves**
- A checkpoint file exists
- A concrete byte `offset` is recorded
- A `saved_at` timestamp shows when the checkpoint was written

**Does NOT prove**
- That all persisted events were processed
- That restarts are idempotent across multiple processes
- That flags or validations were correct

Those claims are evaluated using flags and validation artifacts.


## Section 4 — Contract Enforcement

### Explanation

This section inspects the **contract evidence** to prove that all ingested events
were validated against a **fixed, versioned contract** and that violations
fail fast with no coercion.

The claim being proven:

- A specific contract (name + version) governed ingestion
- Events were either accepted as-is or rejected (no mutation)
- No contract violations occurred in this run

Artifacts under inspection:

- `logs/run.json`
- `outputs/reports/validation_report.json`

What this section **proves**:
- Contract identity and version are explicit
- Contract validation outcomes are recorded

What this section **does not prove**:
- Semantic correctness of the contract rules
- That the contract is sufficient for all compliance needs


In [19]:
# Section 4 — Contract

import json
import pandas as pd

with open("logs/run_ddff9dc1c1b4446bbfbf93aa76a9aceb.json") as f:
    run_log = json.load(f)

with open("outputs/reports/validation_report.json") as f:
    validation = json.load(f)

contract_rows = [
    {"source": "run_log", "key": "contract.name", "value": run_log["contract"]["name"]},
    {"source": "run_log", "key": "contract.version", "value": run_log["contract"]["version"]},
    {"source": "validation", "key": "status", "value": validation["status"]},
    {"source": "validation", "key": "contract_failures", "value": validation["counts"]["contract_failures"]},
]

pd.DataFrame(contract_rows)


,source,key,value
0,run_log,contract.name,money_flow_event_contract
1,run_log,contract.version,v1
2,validation,status,success
3,validation,contract_failures,0


### Evidence

The artifacts confirm that ingestion was governed by a **fixed, versioned contract**
and that no contract violations occurred during this run.

**Proves**
- Contract name and version are explicit and consistent
- Contract validation was enforced
- Zero contract failures were recorded

**Does NOT prove**
- That the contract rules are exhaustive
- That future schema changes are backward compatible

Contract enforcement is therefore proven **for this run only**.


## Section 5 — Flags

### Explanation

This section inspects the **flags artifact** to show the **observable outputs**
of rule evaluation.

The claim being proven:

- Rule breaches, when detected, are emitted as persisted flag records
- Each flag records the rule name, triggering metric, and event context

Artifact under inspection:

- `outputs/flags/flags.log`

What this section **proves**:
- Rule outputs are materialised and persisted
- Flags contain sufficient context for audit and review

What this section **does not prove**:
- Correctness of rule logic
- Completeness of rule coverage
- Absence of missed breaches

Those are evaluated via validation summaries, not individual flags.


In [23]:
# Section 5 — Flags

import pandas as pd

df = pd.read_json("outputs/flags/flags.log", lines=True, nrows=5)
df


,user_id,rule_name,metric_value,threshold,window,event_id,event_time,ingest_time
0,user_08626,threshold,623.03,500,0,b390b956-d176-4c4d-a5f4-62cf992a993e,2026-01-31 20:32:01.260460+00:00,2026-01-31 20:32:01.260471+00:00
1,user_05324,threshold,597.47,500,0,e86df5a7-17b7-4f71-b280-1df6dcad72e1,2026-01-31 20:32:01.819662+00:00,2026-01-31 20:32:01.819673+00:00
2,user_00667,threshold,601.87,500,0,21dc14d8-74f3-49b0-a248-3dc7e0af16ac,2026-01-31 20:32:01.889691+00:00,2026-01-31 20:32:01.889710+00:00
3,user_05325,threshold,740.93,500,0,8d7d9646-41ce-46a0-968c-81bc86e2e84b,2026-01-31 20:32:01.950853+00:00,2026-01-31 20:32:01.950863+00:00
4,user_01729,threshold,751.85,500,0,7896c5ad-166a-4c09-b3ed-c666c64c1c7a,2026-01-31 20:32:01.990301+00:00,2026-01-31 20:32:01.990309+00:00


### Evidence

The displayed records confirm that **rule breaches are emitted as persisted flag events**.

**Proves**
- `flags.log` exists and is readable
- Flags are structured JSON records
- Flags include rule name, metric value, threshold/window, and event identifiers/timestamps

**Does NOT prove**
- That all breaches were detected
- That thresholds/windows are correct
- That outputs are deduplicated across runs

Those are addressed by policy and validation evidence.


## Section 6 — Validation Report

### Explanation

This section inspects the **validation report** to provide a consolidated,
post-run summary of system behaviour.

The claim being proven:

- Validation outcomes were recorded at run completion
- Ingestion, ordering, duplication, and contract policies were enforced
- Aggregate counts are consistent with earlier artifacts

Artifact under inspection:

- `outputs/reports/validation_report.json`

What this section **proves**:
- System-level validation results exist
- Policy enforcement outcomes are explicitly recorded

What this section **does not prove**:
- Internal implementation correctness
- That policies are sufficient for all compliance needs


In [25]:
# Section 6 — Validation Report

import json
import pandas as pd

with open("outputs/reports/validation_report.json") as f:
    report = json.load(f)

rows = [
    {"key": "status", "value": report["status"]},
    {"key": "contract.name", "value": report["contract"]["name"]},
    {"key": "contract.version", "value": report["contract"]["version"]},
    {"key": "events_read", "value": report["counts"]["events_read"]},
    {"key": "events_valid", "value": report["counts"]["events_valid"]},
    {"key": "flags_emitted", "value": report["counts"]["flags_emitted"]},
    {"key": "contract_failures", "value": report["counts"]["contract_failures"]},
    {"key": "duplicate_failures", "value": report["counts"]["duplicate_failures"]},
    {"key": "ordering_failures", "value": report["counts"]["ordering_failures"]},
    {"key": "checkpoint.offset", "value": report["checkpoint"]["offset"]},
]

pd.DataFrame(rows)


,key,value
0,status,success
1,contract.name,money_flow_event_contract
2,contract.version,v1
3,events_read,788
4,events_valid,788
5,flags_emitted,153
6,contract_failures,0
7,duplicate_failures,0
8,ordering_failures,0
9,checkpoint.offset,8299535


### Evidence

The validation report confirms that the system completed successfully and
recorded explicit enforcement outcomes.

**Proves**
- Validation completed with status `success`
- Contract enforcement produced zero failures
- Ordering and duplicate policies produced zero failures
- Aggregate counts and checkpoint offset are explicitly recorded

**Does NOT prove**
- That every possible violation type was exercised
- That validation logic is complete or optimal

This concludes the evidence notebook.


## Section 7 — No-skip (Persisted Events)

### Explanation

This section proves the **no-skip guarantee within the persisted boundary** by comparing:

- the watcher’s final processed byte `offset`
- the physical size of the persisted stream file in bytes

Claim being proven:

- At watcher shutdown, `final_offset` (and checkpoint `offset`) equals the stream file size in bytes,
  implying the watcher consumed the persisted stream **through end-of-file**.

Artifacts under inspection:

- `logs/run.json` (final_offset, stream_path)
- `outputs/checkpoints/state.json` (offset)
- `data/stream.log` (file size in bytes)

Pass criteria:

- `final_offset == checkpoint.offset == stream_file_size_bytes`

Fail criteria:

- Any mismatch between these three values


In [4]:
# Section 7 — No-skip, offsets vs stream file size

import json
import os
import pandas as pd

with open("logs/run_356634e5157044e2b1e32ea004f86121.json") as f:
    run_log = json.load(f)

with open("outputs/checkpoints/state.json") as f:
    checkpoint = json.load(f)

stream_path = run_log["stream_path"]
stream_size_bytes = os.path.getsize(stream_path)

rows = [
    {"metric": "stream_path", "value": stream_path},
    {"metric": "stream_file_size_bytes", "value": stream_size_bytes},
    {"metric": "run.final_offset", "value": run_log["final_offset"]},
    {"metric": "checkpoint.offset", "value": checkpoint["offset"]},
]

pd.DataFrame(rows)


,metric,value
0,stream_path,data/stream.log
1,stream_file_size_bytes,8847160
2,run.final_offset,8847160
3,checkpoint.offset,8847160


### Evidence

The table shows:

- `run.final_offset == checkpoint.offset == stream_file_size_bytes`

the watcher processed the persisted stream **through end-of-file**, and no persisted bytes were skipped within this run boundary.

**Proves**
- End-of-file consumption of the persisted stream at shutdown (byte-level)

**Does NOT prove**
- That pre-persistence (in-memory) events were not lost
- Exactly-once guarantees across multiple independent watcher processes


## Section 8 — Restart-resume (Proven via Checkpoints)

### Explanation

This section proves restart-resume correctness at the byte-offset level using run logs and checkpoint snapshots.

Claim being proven:

- Run A persisted a checkpoint offset at shutdown.
- Run B continued processing from that persisted position and advanced beyond it.

Artifacts under inspection:

- `logs/run_A.json`
- `logs/run_B.json`
- `outputs/checkpoints/state_A.json`
- `outputs/checkpoints/state_B.json`

Pass criteria:

- `run_A.final_offset == state_A.offset`
- `state_B.offset >= state_A.offset`

What this section does **not** claim:

- No duplicated flags across runs
- Exactly-once outputs across restarts


In [8]:
# Section 8 — Restart-resume


import json
import pandas as pd

with open("logs/run_A.json") as f:
    run_a = json.load(f)

with open("logs/run_B.json") as f:
    run_b = json.load(f)

with open("outputs/checkpoints/state_A.json") as f:
    state_a = json.load(f)

with open("outputs/checkpoints/state_B.json") as f:
    state_b = json.load(f)

rows = [
    {"metric": "run_A.run_id", "value": run_a.get("run_id")},
    {"metric": "run_A.final_offset", "value": run_a.get("final_offset")},
    {"metric": "state_A.offset", "value": state_a.get("offset")},
    {"metric": "state_A.saved_at", "value": state_a.get("saved_at")},
    {"metric": "run_B.run_id", "value": run_b.get("run_id")},
    {"metric": "state_B.offset", "value": state_b.get("offset")},
    {"metric": "state_B.saved_at", "value": state_b.get("saved_at")},
    {"metric": "delta_B_minus_A", "value": state_b.get("offset") - state_a.get("offset")},
]

pd.DataFrame(rows)


,metric,value
0,run_A.run_id,854fa2c951894cf0b1f908a240d3845f
1,run_A.final_offset,9796140
2,state_A.offset,9796140
3,state_A.saved_at,2026-02-01T13:49:57.974483+00:00
4,run_B.run_id,354efaba307a4bd480cc14de60daf50a
5,state_B.offset,10600248
6,state_B.saved_at,2026-02-01T13:50:56.294986+00:00
7,delta_B_minus_A,804108


### Evidence

**PASS** as:
- `run_A.final_offset == state_A.offset`
- `state_B.offset >= state_A.offset`
- `delta_B_minus_A` is `>= 0`

This demonstrates that Run A checkpointed its processed position and Run B progressed from that persisted boundary.

**Does NOT prove**
- Exactly-once outputs across runs
- No duplicated flags across restarts


## Section 9 — Determinism (Using Hash)

### Explanation

This section proves deterministic behaviour by comparing hashes of the persisted
flags output from two runs executed against the **same fixed input stream** and
the same contract/rule configuration.

Claim being proven:

- Given identical persisted input and configuration, the emitted `flags.log`
  content is identical across two runs.

Artifacts under inspection:

- `outputs/flags/flags_D1.log`
- `outputs/flags/flags_D2.log`

Pass criteria:

- `sha256(flags_D1.log) == sha256(flags_D2.log)`

What this section does **not** prove:

- Determinism under concurrent writes (live stream)
- Determinism across different inputs or configs


In [11]:
# Section 9 — Determinism

import hashlib
import pandas as pd

def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read())
    return h.hexdigest()

p1 = "outputs/flags/flags_D1.log"
p2 = "outputs/flags/flags_D2.log"

rows = [
    {"artifact": p1, "sha256": sha256_file(p1)},
    {"artifact": p2, "sha256": sha256_file(p2)},
]

print(pd.DataFrame(rows))


# Determinism verdict (exact match)
h1 = sha256_file("outputs/flags/flags_D1.log")
h2 = sha256_file("outputs/flags/flags_D2.log")

import pandas as pd
pd.DataFrame([{"deterministic_flags_hash_match": h1 == h2}])


                     artifact  \
0  outputs/flags/flags_D1.log   
1  outputs/flags/flags_D2.log   

                                              sha256  
0  f6dd8ed60af67a3a4c85149b79379ef8d898a19a1fc96a...  
1  f6dd8ed60af67a3a4c85149b79379ef8d898a19a1fc96a...  


,deterministic_flags_hash_match
0,True


### Evidence

**PASS** as SHA256 hashes match.

This demonstrates that, for the fixed persisted input and unchanged configuration,
the emitted flags output is deterministic at the file-content level.

**Does NOT prove**
- Determinism during live ingestion
- Determinism if the input stream differs


## Section 10 — Fail-fast Contract (Negative Test)

### Explanation

This section proves **fail-fast behaviour** when an invalid record is present
in the persisted input stream.

An invalid line was **intentionally appended** to the existing persisted
stream file to simulate upstream corruption or malformed input.

Claim being proven:

- The watcher fails immediately on encountering an invalid record
- The run is marked as failed
- No further processing or flag emission continues after the failure

Artifact under inspection:

- `logs/run_bad.json`
- `outputs/reports/validation_bad.json`
- `outputs/flags/flags_bad.log` (if created)

This negative test is **isolated** and does not affect other sections.


In [19]:
# Section 10 — Fail-fast Contract

import json
import pandas as pd
from pathlib import Path

with open("logs/run_bad.json") as f:
    run_bad = json.load(f)

with open("outputs/reports/validation_bad.json") as f:
    validation_bad = json.load(f)

flags_bad_path = Path("outputs/flags/flags_bad.log")
flags_bad_lines = 0
if flags_bad_path.exists():
    with open(flags_bad_path) as f:
        flags_bad_lines = sum(1 for _ in f)

rows = [
    {"metric": "run.status", "value": run_bad.get("status")},
    {"metric": "run.events_read", "value": run_bad.get("counts", {}).get("events_read")},
    {"metric": "run.flags_emitted", "value": run_bad.get("counts", {}).get("flags_emitted")},
    {"metric": "validation.status", "value": validation_bad.get("status")},
    {"metric": "contract_failures", "value": validation_bad.get("counts", {}).get("contract_failures")},
    {"metric": "json_parse_failures", "value": validation_bad.get("counts", {}).get("json_parse_failures")},
    {"metric": "flags_file_exists", "value": flags_bad_path.exists()},
    {"metric": "flags_line_count", "value": flags_bad_lines},
]

pd.DataFrame(rows)


,metric,value
0,run.status,failed
1,run.events_read,1
2,run.flags_emitted,0
3,validation.status,failed
4,contract_failures,0
5,json_parse_failures,1
6,flags_file_exists,True
7,flags_line_count,0


### Evidence

The validation report shows that the watcher failed immediately upon encountering
an invalid record.

Observed:

- A JSON parse failure was recorded (`json_parse_failures = 1`)
- Validation status is `failed`
- Flags were not emitted

**PASS condition**

- The run fails on the first invalid record
- No recovery, coercion, or continued processing occurs after the failure

**Clarification**

- Failure occurs at first invalid record; no recovery/coercion is applied.
- The system does not guarantee “zero outputs” for a run containing invalid data.

This confirms **fail-fast behaviour at the point of contract violation**.
